# Prueba de BioEmu-2 en Colab

**EIP, Bloques 5 y 6 · clase 3 (6.2) · 04/11/2026**

Objetivo: comprobar si BioEmu-2 **cabe en la tarjeta T4 de Colab gratuito**, porque de eso
depende que los alumnos puedan generar conjuntos de conformaciones ellos mismos o haya que
precalcularlos en el clúster.

Lo que se sabe antes de probar:

- El repositorio de BioEmu2 lista la **T4 entre las tarjetas probadas**, pero no publica el
  consumo de memoria.
- La ficha del modelo menciona una A100 de 80 GB: es el equipo de las **pruebas de
  rendimiento**, no un mínimo.
- Un despliegue de terceros declara un presupuesto de **16 GiB de VRAM** para sistemas de
  hasta **448 residuos**. Una T4 tiene 16 GB, de los que quedan unos 15 libres.

Las seis dianas del bloque caben en ese límite: alfa-sinucleína 140, adenilato quinasa 214,
GFP 238 y p53 393 residuos.

> ⚠️ **Antes de ejecutar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU.*
> Sin eso, la primera celda avisa y no tiene sentido seguir.

In [ ]:
# 1. Qué tarjeta nos ha tocado y cuánta memoria tiene
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Instalación. Tarda unos minutos: descarga el modelo y sus dependencias.
#    El extra [cuda] es obligatorio en tarjetas de capacidad 7.0 y 7.5, que es el caso de la T4.
#    En PyPI está como bioemu2 (0.1.0rc0 el 04/11/2026); si fallara, la alternativa es
#    clonar https://github.com/microsoft/bioemu2 e instalar con  pip install ".[cuda]"
!python -m pip install -q "bioemu2[cuda]"
!bioemu2 --help

In [ ]:
# 3. Caso corto primero: la adenilato quinasa (214 residuos), que es la diana de los dos
#    estados de la clase 3 y el ejemplo que usa la propia documentación.
#    Si esto no cabe en la T4, no cabrá nada.
import time

t0 = time.time()
!bioemu2 sample-equilibrium --pdb-id 1AKE --out /content/salida_adk
print(f"\nTardó {time.time() - t0:.0f} s")

In [ ]:
# 4. Qué ha dejado y cuánta memoria de la tarjeta se llegó a usar.
#    El pico de memoria es EL dato de esta prueba: dice si cabe con margen o por los pelos.
!ls -la /content/salida_adk
import torch
print(f"\nPico de memoria en la tarjeta: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print(f"Memoria total de la tarjeta:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 5. Si lo anterior ha ido bien, el caso que de verdad interesa para la clase:
#    la alfa-sinucleína (140 residuos), que es la proteína desordenada.
#    Aquí el conjunto SÍ debería ser ancho, al contrario que en una proteína plegada.
ASYN = ("MDVFMKGLSKAKEGVVAAAEKTKQGVAEAAGKTKEGVLYVGSKTKEGVVHGVATVAEKTKEQVTNVGGAVVTGVTAVAQKTVEG"
        "AGSIAAATGFVKKDQLGKNEEGAPQEGILEDMPVDPDNEAYEMPSEEGYQDYEPEA")
print(len(ASYN), "residuos")

!bioemu2 sample-equilibrium --sequence "$ASYN" --out /content/salida_asyn
!ls -la /content/salida_asyn

## Qué anotar al terminar

1. **¿Cabe en la T4?** Sí / no, y con qué pico de memoria.
2. **¿Cuánto tarda** cada caso? De ahí sale si da tiempo en una clase de una hora o hay que
   lanzarlo en la anterior.
3. **Nombres de los ficheros de salida.** La documentación dice `samples.xtc` y `topology.cif`;
   si no coincide, hay que corregir la diapositiva 6.2 del guion.

Con esas tres respuestas se decide si la clase 3 se hace en directo o con conjuntos
precalculados en el clúster.

> La sintaxis exacta de la celda 5 (`--sequence`) está **sin verificar**: la documentación
> muestra el ejemplo con `--pdb-id`. Si falla, mirar `bioemu2 sample-equilibrium --help`.